# Evaluación del Agente de Triaje Inteligente — Casos 1 a 4 (Unidad 5)

Este notebook evalúa cuantitativamente el comportamiento neuro-agéntico del sistema de triaje para reseñas de streaming a través de dos dimensiones fundamentales:

- **Trayectoria:** Verifica si el agente activa exactamente las herramientas esperadas y en el orden estricto de ejecución. La verificación se realiza exclusivamente a partir de los `tool_calls` estructurados (`AIMessage.tool_calls`), sin deducir herramientas a partir de la respuesta final de texto.
- **Resultado:** Evalúa mediante verificadores deterministas basados en código si las salidas técnicas (`ToolMessage`) y la respuesta final cumplen con las políticas corporativas y la veracidad de la inferencia local.

Toda la lógica de evaluación reutilizable reside en `src/evaluacion.py`.

In [1]:
import os, sys, json, re
from importlib.metadata import version

sys.path.insert(0, os.path.abspath(".."))
os.chdir(os.path.abspath(".."))  # raíz del repositorio

from dotenv import load_dotenv
load_dotenv()

from src.config import GEMINI_MODEL, RPM_GRATUITO
from src.herramientas import obtener_tools
from src.agente import construir_agente
from src.evaluacion import (
    evaluar_caso,
    imprimir_caso,
    pausa_entre_casos,
    guardar_resultados,
    _texto,
    verificar_caso_1,
    verificar_caso_2,
)

print("=== ENTORNO DE EVALUACIÓN ===")
for pkg in ["langchain", "langgraph", "langchain-google-genai"]:
    print(f"- {pkg}: {version(pkg)}")
print(f"- GEMINI_MODEL: {GEMINI_MODEL}")
print(f"- GOOGLE_API_KEY configurada: {bool(os.getenv('GOOGLE_API_KEY'))}")
print(f"- RPM_GRATUITO: {RPM_GRATUITO}")

tools = obtener_tools()
print(f"\nHerramientas enlazadas ({len(tools)}):")
for t in tools:
    print(f"\n* {t.name}:\n  {t.description.strip()}")

agente = construir_agente(tools)
resultados = []
print("\n[OK] Agente compilado y listo para evaluación.")

=== ENTORNO DE EVALUACIÓN ===
- langchain: 1.5.0
- langgraph: 1.2.22
- langchain-google-genai: 4.5.0
- GEMINI_MODEL: gemini-3.5-flash-lite
- GOOGLE_API_KEY configurada: True
- RPM_GRATUITO: 15

Herramientas enlazadas (2):

* clasificar_sentimiento_resena:
  Ejecuta la red neuronal local entrenada con reseñas IMDB en inglés y clasifica su sentimiento.

Úsala cuando el usuario entregue el texto de una reseña y pida clasificarla o hacer triaje.
NO la uses para saludos, preguntas sobre políticas ni para un texto ya clasificado en este turno.

Args:
    texto: la reseña literal y completa, sin resumir ni traducir.

Returns:
    JSON de una línea con sentimiento ("positivo"/"negativo"), confianza (0-1) y baja_confianza (bool).

* consultar_politica_triaje:
  Consulta la politica interna y protocolo de respuesta para una resena de cliente.

Usala cuando conozcas el sentimiento/confianza de la resena o cuando el usuario pregunte
directamente sobre lineamientos, politicas de triaje o protocolos

## Caso 1
Consulta general o saludo sin invocar herramientas.
- **Entrada:** `"Hola, ¿qué puedes hacer por mí?"`
- **Herramientas esperadas:** `[]`
- **Verificador:** Respuesta no vacía, 0 `ToolMessage` y sin porcentajes o métricas inventadas.

In [2]:
r1 = evaluar_caso(
    agente,
    "Caso 1",
    "Hola, ¿qué puedes hacer por mí?",
    [],
    verificar_caso_1
)
imprimir_caso(r1)
resultados.append(r1)

C:\Users\USUARIO\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Caso: Caso 1
Entrada: Hola, ¿qué puedes hacer por mí?
Herramientas esperadas: []
Herramientas invocadas: []
Trayectoria: CUMPLE
Resultado: CUMPLE (respuesta no vacía, sin ToolMessage y sin clasificación inventada)
Respuesta final: ¡Hola! Puedo ayudarte a clasificar reseñas de streaming (positivas o negativas) usando un modelo local y consultar las políticas internas de triaje y protocolo de respuesta para cada caso. 

Puedes enviarme el texto de una reseña en inglés para comenzar.
Tokens: 609


## Caso 2
Consulta directa de protocolo/política sin clasificación de texto.
- **Entrada:** `"¿Qué política debo aplicar si un cliente amenaza con acciones legales en su reseña?"`
- **Herramientas esperadas:** `["consultar_politica_triaje"]`
- **Verificador:** Identificador de política (`POL_NEG`) o acción del `ToolMessage` presentes en la respuesta final.

In [3]:
print(f"Pausa entre turnos: {round(pausa_entre_casos())} s")
r2 = evaluar_caso(
    agente,
    "Caso 2",
    "¿Qué política debo aplicar si un cliente amenaza con acciones legales en su reseña?",
    ["consultar_politica_triaje"],
    verificar_caso_2
)
imprimir_caso(r2)
resultados.append(r2)

Pausa entre turnos: 5 s


C:\Users\USUARIO\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


C:\Users\USUARIO\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Caso: Caso 2
Entrada: ¿Qué política debo aplicar si un cliente amenaza con acciones legales en su reseña?
Herramientas esperadas: ['consultar_politica_triaje']
Herramientas invocadas: ['consultar_politica_triaje']
Trayectoria: CUMPLE
Resultado: CUMPLE (pol_id 'POL_NEG' del ToolMessage aparece en la respuesta)
Respuesta final: Para una reseña con amenaza de acciones legales, aplica la política **POL_NEG**:

- **Sentimiento:** Negativo
- **Confianza:** 1.0
- **Prioridad:** ALTA
- **Acción:** Ticket de insatisfacción (escalamiento legal/soporte senior)
- **Pauta:** Empatizar con el malestar, agradecer la franqueza y ofrecer soporte directo evitando confrontaciones.
Tokens: 1365


## Caso 3
Caso completo con clasificación de reseña negativa real y consulta de protocolo corporativo.
- **Origen:** Split de test real de IMDB (`idx_test = 41`).
- **Herramientas esperadas:** `["clasificar_sentimiento_resena", "consultar_politica_triaje"]` en orden.
- **Verificador:** Sentimiento de Tool #1 y política/acción de Tool #2 presentes en la respuesta final.

In [4]:
def verificar_caso_3(respuesta: str, tool_msgs) -> tuple:
    if len(tool_msgs) < 2:
        return False, f"Se esperaban 2 ToolMessage, se obtuvieron {len(tool_msgs)}"
    try:
        salida_t1 = json.loads(_texto(tool_msgs[0].content))
        salida_t2 = json.loads(_texto(tool_msgs[1].content))
    except Exception as e:
        return False, f"Error al parsear ToolMessages como JSON: {e}"
    # (a) Respuesta menciona el sentimiento de Tool #1
    sent = salida_t1.get("sentimiento", "").lower()
    if sent not in respuesta.lower():
        return False, f"El sentimiento de Tool #1 (\"{sent}\") no aparece en la respuesta final"
    # (b) Respuesta incluye la accion o pol_id de Tool #2
    norm = lambda s: re.sub(r"[\s_\-]+", "", str(s).lower())
    resp_n = norm(respuesta)
    pol_id = salida_t2.get("pol_id", "")
    accion = salida_t2.get("accion", "")
    if not (norm(pol_id) in resp_n or norm(accion) in resp_n):
        return False, f"Ni pol_id (\"{pol_id}\") ni accion (\"{accion}\") de Tool #2 aparecen en la respuesta"
    return True, f"Sentimiento \"{sent}\" y politica/accion ({pol_id}/{accion}) presentes en respuesta"

idx_test = 41
resena_test_negativa = "Widow hires a psychopath as a handyman. Sloppy film noir thriller which doesn't make much of its tension promising set-up. (3/10)"
print(f"Índice del split test IMDB evaluado: {idx_test}")
print(f"Pausa entre turnos: {round(pausa_entre_casos())} s")

entrada_c3 = f'Haz el triaje de esta reseña de un cliente: "{resena_test_negativa}"'
r3 = evaluar_caso(
    agente,
    "Caso 3",
    entrada_c3,
    ["clasificar_sentimiento_resena", "consultar_politica_triaje"],
    verificar_caso_3
)
imprimir_caso(r3)
resultados.append(r3)

Índice del split test IMDB evaluado: 41


Pausa entre turnos: 5 s


C:\Users\USUARIO\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


C:\Users\USUARIO\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


C:\Users\USUARIO\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Caso: Caso 3
Entrada: Haz el triaje de esta reseña de un cliente: "Widow hires a psychopath as a handyman. Sloppy film noir thriller which doesn't make much of its tension promising set-up. (3/10)"
Herramientas esperadas: ['clasificar_sentimiento_resena', 'consultar_politica_triaje']
Herramientas invocadas: ['clasificar_sentimiento_resena', 'consultar_politica_triaje']
Trayectoria: CUMPLE
Resultado: CUMPLE (Sentimiento "negativo" y politica/accion (POL_NEG/Ticket de insatisfaccion) presentes en respuesta)
Respuesta final: - **Sentimiento:** Negativo
- **Confianza:** 1.0 (100%)
- **Política ID:** POL_NEG - **Prioridad:** ALTA
- **Acción:** Ticket de insatisfacción
- **Pauta:** Empatizar con malestar, agradecer franqueza y ofrecer soporte directo.
Tokens: 2230


## Caso 4
Caso completo con el fallo real / baja confianza extraído de `assets/apuntes/fallo_real.md` (Caso B).
- **Herramientas esperadas:** `["clasificar_sentimiento_resena", "consultar_politica_triaje"]` en orden.
- **Verificador:** Certeza de Tool #1 coincide con `fallo_real.md` (0.5835 ± 1e-4) y la respuesta final comunica baja confianza y derivación a moderación / revisión humana.

In [5]:
def verificar_caso_4(respuesta: str, tool_msgs) -> tuple:
    if len(tool_msgs) < 2:
        return False, f"Se esperaban 2 ToolMessage, se obtuvieron {len(tool_msgs)}"
    try:
        salida_t1 = json.loads(_texto(tool_msgs[0].content))
        salida_t2 = json.loads(_texto(tool_msgs[1].content))
    except Exception as e:
        return False, f"Error al parsear ToolMessages como JSON: {e}"
    # Validar coincidencia de confianza con tolerancia 1e-4
    conf_medida = float(salida_t1.get("confianza", 0.0))
    conf_esperada = 0.5835
    if abs(conf_medida - conf_esperada) > 1e-4:
        return False, f"Confianza medida ({conf_medida}) difiere de la esperada ({conf_esperada}) > 1e-4"
    # Validar comunicacion de baja confianza y revision humana
    norm = lambda s: re.sub(r"[\s_\-]+", "", str(s).lower())
    resp_n = norm(respuesta)
    accion = salida_t2.get("accion", "")
    pol_id = salida_t2.get("pol_id", "")
    senales_revision = ["bajaconfianza", "moderacion", "revisionhumana", norm(pol_id), norm(accion)]
    if not any(s in resp_n for s in senales_revision):
        return False, f"La respuesta no comunica baja confianza ni revision humana/moderacion"
    return True, f"Confianza {conf_medida} coincide con fallo_real.md y revision humana comunicada"

# Leer texto del Caso B directamente de assets/apuntes/fallo_real.md
with open("assets/apuntes/fallo_real.md", encoding="utf-8") as f:
    contenido_fallo = f.read()

match_texto = re.search(r"Texto completo.*?:[\s\*]*`([^`]+)`", contenido_fallo.split("Caso B")[1])
assert match_texto, "No se encontró el texto de Caso B en assets/apuntes/fallo_real.md"
texto_caso_b = match_texto.group(1).strip()
print("Texto Caso B extraído de fallo_real.md:", repr(texto_caso_b))
print(f"Pausa entre turnos: {round(pausa_entre_casos())} s")

entrada_c4 = f'Haz el triaje de esta reseña de un cliente: "{texto_caso_b}"'
r4 = evaluar_caso(
    agente,
    "Caso 4",
    entrada_c4,
    ["clasificar_sentimiento_resena", "consultar_politica_triaje"],
    verificar_caso_4
)
imprimir_caso(r4)
resultados.append(r4)

Texto Caso B extraído de fallo_real.md: 'Not bad, but not great either. It sits right in the middle.'


Pausa entre turnos: 5 s


C:\Users\USUARIO\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


C:\Users\USUARIO\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


C:\Users\USUARIO\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Caso: Caso 4
Entrada: Haz el triaje de esta reseña de un cliente: "Not bad, but not great either. It sits right in the middle."
Herramientas esperadas: ['clasificar_sentimiento_resena', 'consultar_politica_triaje']
Herramientas invocadas: ['clasificar_sentimiento_resena', 'consultar_politica_triaje']
Trayectoria: CUMPLE
Resultado: CUMPLE (Confianza 0.5835 coincide con fallo_real.md y revision humana comunicada)
Respuesta final: - **Sentimiento:** Negativo
- **Confianza:** 0.58 (Baja confianza < 0.70)
- **Política aplicada:** POL_REV
- **Prioridad:** MEDIA
- **Acción:** Derivar a moderación
- **Pauta:** Solicitar cortésmente mayor detalle o clarificación al usuario.
Tokens: 2130


> [!IMPORTANT]
> **Principio de Evaluación de Trayectoria (Requisito 9):**  
> La trayectoria del agente se obtiene y valida **estrictamente a partir de los `tool_calls`** estructurados contenidos en los mensajes conversacionales (`AIMessage.tool_calls`), y jamás analizando o buscando palabras clave en la respuesta final de texto.  
> Esta inspección es realizada por la función `herramientas_invocadas` en `src/agente.py` y consumida de forma determinista por `evaluar_caso` en `src/evaluacion.py`.

## Matriz de evaluación
Resumen estructurado de desempeño para los cuatro casos analizados.

In [6]:
import pandas as pd

cumple = lambda ok: "CUMPLE" if ok else "NO CUMPLE"

filas_df = []
for r in resultados:
    entrada_resumida = r["entrada"][:50] + ("..." if len(r["entrada"]) > 50 else "")
    filas_df.append({
        "caso": r["nombre"],
        "entrada resumida": entrada_resumida,
        "herramientas esperadas": ", ".join(r["esperadas"]) if r["esperadas"] else "[]",
        "herramientas invocadas": ", ".join(r["invocadas"]) if r["invocadas"] else "[]",
        "trayectoria": cumple(r["trayectoria_ok"]),
        "resultado": cumple(r["resultado_ok"]),
        "tokens": r["tokens"],
    })

df_matriz = pd.DataFrame(filas_df)

# Guardar resultados JSON y matriz Markdown
guardar_resultados(resultados, "assets/apuntes/resultados_evaluacion.json")
with open("assets/apuntes/matriz_evaluacion.md", "w", encoding="utf-8") as f:
    f.write("# Matriz de Evaluación del Agente de Triaje\n\n")
    f.write(df_matriz.to_markdown(index=False))
    f.write("\n")

total_casos = len(resultados)
n_trayectoria_ok = sum(1 for r in resultados if r["trayectoria_ok"])
n_resultado_ok = sum(1 for r in resultados if r["resultado_ok"])

print(f"\nResumen cuantitativo global:")
print(f"- Casos con trayectoria correcta sobre el total: {n_trayectoria_ok} de {total_casos} ({100.0 * n_trayectoria_ok / total_casos:.1f}%)")
print(f"- Casos con resultado correcto sobre el total   : {n_resultado_ok} de {total_casos} ({100.0 * n_resultado_ok / total_casos:.1f}%)\n")

df_matriz


Resumen cuantitativo global:
- Casos con trayectoria correcta sobre el total: 4 de 4 (100.0%)
- Casos con resultado correcto sobre el total   : 4 de 4 (100.0%)



,caso,entrada resumida,herramientas esperadas,herramientas invocadas,trayectoria,resultado,tokens
0,Caso 1,"Hola, ¿qué puedes hacer por mí?",[],[],CUMPLE,CUMPLE,609
1,Caso 2,¿Qué política debo aplicar si un cliente amena...,consultar_politica_triaje,consultar_politica_triaje,CUMPLE,CUMPLE,1365
2,Caso 3,"Haz el triaje de esta reseña de un cliente: ""W...","clasificar_sentimiento_resena, consultar_polit...","clasificar_sentimiento_resena, consultar_polit...",CUMPLE,CUMPLE,2230
3,Caso 4,"Haz el triaje de esta reseña de un cliente: ""N...","clasificar_sentimiento_resena, consultar_polit...","clasificar_sentimiento_resena, consultar_polit...",CUMPLE,CUMPLE,2130
